In [21]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from featureEngineer import engineer_features
import warnings

from pathlib import Path
import sys
import os
from datetime import datetime
from dataScraper import *

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import nameDict
warnings.filterwarnings("ignore")

In [2]:
pd.set_option('display.max_columns', None)

df = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"])
df = df.sort_values(["PLAYER_ID", "GAME_DATE"]).reset_index(drop=True)
 
# Derived features
df['PF_PER_MIN'] = df['PF'] / df['MIN'].replace(0, np.nan)
df["PTS_PER_MIN"] = df["PTS"] / df["MIN"].replace(0, np.nan)
df["IS_HOME"]     = df["MATCHUP"].str.contains("vs\\.").astype(int)
df["SPREAD_PROXY"] = df["TEAM_PLUS_MINUS"]  # actual outcome; use betting spread at prediction time
df['STARTING'] = df['START_POSITION'].notna().astype(int)
df = engineer_features(df)
df['TEAM_MIN_RANK_L10'] = (df.groupby(['TEAM_ID','GAME_DATE'])['MIN_roll10'].rank(ascending=False, method='dense'))
df['TEAM_USG_RANK_L10'] = (df.groupby(['TEAM_ID','GAME_DATE'])['USG_PCT_roll10'].rank(ascending=False, method='dense'))
df['MEDIAN_MIN_ROLLING_10'] = (df.groupby('PLAYER_ID')['MIN'].transform(lambda x: x.shift(1).rolling(10).median().round(2)))
df["PTS_PER_MIN_10_ewm"] = df.groupby("PLAYER_ID")["PTS_PER_MIN"].transform(lambda x: x.shift(1).ewm(span=10).mean())
df['PTS_ewm10'] = (df.groupby('PLAYER_ID')['PTS'].transform(lambda x: x.shift(1).ewm(span=10, adjust=False).mean()))
df['MIN_ewm10'] = (df.groupby('PLAYER_ID')['MIN'].transform(lambda x: x.shift(1).ewm(span=10, adjust=False).mean()))
df["GP"] = df.groupby("PLAYER_ID").cumcount()  # 0-indexed; GP=0 means first game
df["BLOWOUT_RISK"] = df.groupby("PLAYER_ID")["TEAM_PLUS_MINUS"].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).std())
df['MIN_TREND'] = df['MIN_roll3'] - df['MIN_roll10']
df[df['PLAYER_ID'] == 2544].tail()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,START_POSITION,PF_PER_MIN,PTS_PER_MIN,IS_HOME,SPREAD_PROXY,STARTING,MIN_roll3,PTS_roll3,USG_PCT_roll3,PF_roll3,PLUS_MINUS_roll3,POSS_roll3,PTS_PER_MIN_roll3,PF_PER_MIN_roll3,TS_PCT_roll3,AST_TO_roll3,NET_RATING_roll3,MIN_roll5,PTS_roll5,USG_PCT_roll5,PF_roll5,PLUS_MINUS_roll5,POSS_roll5,PTS_PER_MIN_roll5,PF_PER_MIN_roll5,TS_PCT_roll5,AST_TO_roll5,NET_RATING_roll5,MIN_roll10,PTS_roll10,USG_PCT_roll10,PF_roll10,PLUS_MINUS_roll10,POSS_roll10,PTS_PER_MIN_roll10,PF_PER_MIN_roll10,TS_PCT_roll10,AST_TO_roll10,NET_RATING_roll10,MIN_lag1,PTS_lag1,PLUS_MINUS_lag1,PIE_lag1,STARTING_lag1,POSS_lag1,MIN_lag2,PTS_lag2,PLUS_MINUS_lag2,PIE_lag2,STARTING_lag2,POSS_lag2,STARTER_ROLL10_PCT,MIN_season_avg,PTS_season_avg,USG_PCT_season_avg,POSS_season_avg,PF_season_avg,OFF_RATING_season_avg,DEF_RATING_season_avg,TEAM_PACE_roll5,TEAM_PTS_roll5,TEAM_NET_RATING_roll5,MIN_share_proxy,PTS_share_proxy,TEAM_POSS_roll5,TEAM_POSS_share,DAYS_REST,IS_B2B,GAME_NUMBER,MIN_std10,TEAM_MIN_RANK_L10,TEAM_USG_RANK_L10,MEDIAN_MIN_ROLLING_10,PTS_PER_MIN_10_ewm,PTS_ewm10,MIN_ewm10,GP,BLOWOUT_RISK,MIN_TREND
44,1320,2025-26,2544,LeBron James,LeBron,1610612747,LAL,Los Angeles Lakers,22500960,2026-03-12,LAL vs. CHI,W,33.233333,7,13,0.538,0,2,0.000,4,6,0.667,2,5,7,7,4,2,1,1,1,5,18,12,41.9,0,0,38.0,1,33:14,1,132.6,127.4,127.4,111.2,114.1,114.1,21.4,13.3,13.3,0.241,1.75,26.9,0.051,0.143,0.095,15.4,15.0,0.538,0.575,0.218,0.239,103.27,103.99,86.66,103.99,0.121,73,7.0,13.0,55,99,0.556,17,36,0.472,15,21,0.714,14,27,41,31,10.0,9,4,3,15,18,142,12.0,136.2,137.9,122.6,126.2,13.6,11.7,0.564,3.10,20.3,0.380,0.630,0.500,0.097,0.641,0.656,105.1,103.00,85.83,103,0.531,1610612741,CHI,Chicago Bulls,51,95,0.537,15,36,0.417,13,16,0.813,12,29,41,34,16.0,7,3,4,18,15,130,-12.0,122.6,126.2,136.2,137.9,-13.6,-11.7,0.667,2.13,22.1,0.370,0.620,0.500,0.155,0.616,0.637,105.1,103.00,85.83,103,0.469,F,0.030090,0.541625,1,12.0,1,31.31,20.33,0.25,1.33,2.00,66.33,0.67,0.04,0.68,3.69,7.37,31.39,19.6,0.26,1.0,7.4,64.6,0.624403,0.03,0.64,3.66,14.52,32.78,20.2,0.26,1.1,1.4,67.7,0.62,0.03,0.60,3.41,4.43,33.700000,16.0,-10.0,0.166,1.0,70.0,33.466667,21.0,3.0,0.156,1.0,74.0,1.0,33.10,21.43,0.27,69.27,1.34,113.70,114.05,99.20,116.2,9.02,0.13,0.17,64.6,1.0,7.0,0,44,3.01,3.0,2.0,33.58,0.625067,19.884972,32.355750,44,15.642890,-1.47
45,1015,2025-26,2544,LeBron James,LeBron,1610612747,LAL,Los Angeles Lakers,22500974,2026-03-14,LAL vs. DEN,W,40.216667,7,13

In [3]:
# Drop stale prior columns so re-running this cell does not merge into prior_mean_x / prior_mean_y
_prior_stale = [
    c
    for c in df.columns
    if c in ("prior_mean", "prior_std")
    or c.startswith("prior_mean_")
    or c.startswith("prior_std_")
]
if _prior_stale:
    df = df.drop(columns=_prior_stale)

league_prior = (
    df.groupby("STARTING")["PTS_PER_MIN"]
    .agg(prior_mean="mean", prior_std="std")
    .reset_index()
)
df = df.merge(league_prior, on="STARTING", how="left")

# Conjugate normal–normal posterior mean (vectorized)
prior_m = df["prior_mean"]
prior_v = (df["prior_std"] ** 2).clip(lower=1e-12)

obs_mean = df["PTS_PER_MIN_10_ewm"].where(df["PTS_PER_MIN_10_ewm"].notna(), prior_m)
obs_n = df["GP"].clip(upper=20)
obs_var = prior_v

prior_prec = 1 / prior_v
obs_prec = obs_n / obs_var
posterior_mean = (prior_prec * prior_m + obs_prec * obs_mean) / (prior_prec + obs_prec)

df["BAYES_PTS_PER_MIN"] = np.where(obs_n.to_numpy() == 0, prior_m.to_numpy(), posterior_mean.to_numpy())
df.head()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,START_POSITION,PF_PER_MIN,PTS_PER_MIN,IS_HOME,SPREAD_PROXY,STARTING,MIN_roll3,PTS_roll3,USG_PCT_roll3,PF_roll3,PLUS_MINUS_roll3,POSS_roll3,PTS_PER_MIN_roll3,PF_PER_MIN_roll3,TS_PCT_roll3,AST_TO_roll3,NET_RATING_roll3,MIN_roll5,PTS_roll5,USG_PCT_roll5,PF_roll5,PLUS_MINUS_roll5,POSS_roll5,PTS_PER_MIN_roll5,PF_PER_MIN_roll5,TS_PCT_roll5,AST_TO_roll5,NET_RATING_roll5,MIN_roll10,PTS_roll10,USG_PCT_roll10,PF_roll10,PLUS_MINUS_roll10,POSS_roll10,PTS_PER_MIN_roll10,PF_PER_MIN_roll10,TS_PCT_roll10,AST_TO_roll10,NET_RATING_roll10,MIN_lag1,PTS_lag1,PLUS_MINUS_lag1,PIE_lag1,STARTING_lag1,POSS_lag1,MIN_lag2,PTS_lag2,PLUS_MINUS_lag2,PIE_lag2,STARTING_lag2,POSS_lag2,STARTER_ROLL10_PCT,MIN_season_avg,PTS_season_avg,USG_PCT_season_avg,POSS_season_avg,PF_season_avg,OFF_RATING_season_avg,DEF_RATING_season_avg,TEAM_PACE_roll5,TEAM_PTS_roll5,TEAM_NET_RATING_roll5,MIN_share_proxy,PTS_share_proxy,TEAM_POSS_roll5,TEAM_POSS_share,DAYS_REST,IS_B2B,GAME_NUMBER,MIN_std10,TEAM_MIN_RANK_L10,TEAM_USG_RANK_L10,MEDIAN_MIN_ROLLING_10,PTS_PER_MIN_10_ewm,PTS_ewm10,MIN_ewm10,GP,BLOWOUT_RISK,MIN_TREND,prior_mean,prior_std,BAYES_PTS_PER_MIN
0,18066,2025-26,2544,LeBron James,LeBron,1610612747,LAL,Los Angeles Lakers,22500253,2025-11-18,LAL vs. UTA,W,29.616667,4,7,0.571,2,3,0.667,1,4,0.250,1,2,3,12,1,1,0,0,0,3,11,1,34.6,1,0,30.0,1,29:37,1,133.8,133.8,133.8,123.1,128.4,128.4,10.6,5.5,5.5,0.414,12.0,54.5,0.040,0.083,0.061,4.5,4.6,0.714,0.628,0.135,0.135,109.30,106.97,89.14,106.97,0.110,65,4.0,7.0,50,84,0.595,11,32,0.344,29,40,0.725,11,30,41,31,17.0,9,1,2,21,30,140,14.0,130.1,130.8,114.6,118.9,15.5,12.0,0.620,1.82,20.4,0.350,0.727,0.548,0.159,0.661,0.689,108.8,106.5,88.75,107,0.550,1610612762,UTA,Utah Jazz,48,92,0.522,13,45,0.289,17,18,0.944,8,25,33,33,18.0,13,2,1,30,21,126,-14.0,114.6,118.9,130.1,130.8,-15.5,-12.0,0.688,1.83,21.9,0.273,0.650,0.452,0.170,0.592,0.631,108.8,106.5,88.75,106,0.450,F,0.000000,0.371412,1,14.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,99.5,110.4,0.14,NaN,NaN,NaN,NaN,3.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,0.509133,0.249236,0.509133
1,17269,2025-26,2544,LeBron James,LeBron,1610612747,LAL,Los Angeles Lakers,22500282,2025-11-23,LAL @ UTA,W,34.321667,8,18,0.444,0,4,0.000,1,2,0.500,0,6,6,8,2,1,0,0,2,2,17,-14,37.

In [13]:
MIN_FEATURES = [
    "MIN_roll5",
    "MIN_roll10",
    "MIN_season_avg",
    "MIN_lag1",
    "MIN_ewm10",
    "MIN_TREND",
    "STARTING",
    "STARTING_lag1",
    "STARTER_ROLL10_PCT",
    "BLOWOUT_RISK",
    "DAYS_REST",
    "IS_B2B",
    "IS_HOME",
    'TEAM_MIN_RANK_L10',
    "TEAM_USG_RANK_L10",
    'TEAM_POSS_roll5',
    'TEAM_PACE_roll5',
    "MIN_share_proxy",
    "MEDIAN_MIN_ROLLING_10",
    "MIN_std10",
    "PLUS_MINUS_roll5",
    "PF_roll5",
    "PF_PER_MIN"
]

In [17]:
# Drop rows with NaN in features or target
min_df = df[MIN_FEATURES + ["MIN", "GAME_DATE"]].dropna()

# -- TIME-BASED SPLIT --
split_date = min_df["GAME_DATE"].quantile(0.75)  # train on first 75% of season
train_mask = min_df["GAME_DATE"] <= split_date
val_mask   = ~train_mask

X_train_min = min_df.loc[train_mask, MIN_FEATURES]
y_train_min = min_df.loc[train_mask, "MIN"]
X_val_min   = min_df.loc[val_mask, MIN_FEATURES]
y_val_min   = min_df.loc[val_mask, "MIN"]

xgb_min = XGBRegressor(
    n_estimators=600,
    learning_rate=0.01,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.7,
    reg_alpha=1.0,
    reg_lambda=2.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

xgb_min.fit(
    X_train_min,
    y_train_min,
    eval_set=[(X_val_min, y_val_min)],
    verbose=False,
)

min_preds_val = xgb_min.predict(X_val_min)
min_mae = mean_absolute_error(y_val_min, min_preds_val)
min_rmse = np.sqrt(mean_squared_error(y_val_min, min_preds_val))
min_r2 = r2_score(y_val_min, min_preds_val)

print(f"[Minutes Model] Validation R2: {min_r2:.4f}")
print(f"[Minutes Model] Validation RMSE: {min_rmse:.2f} min")
print(f"[Minutes Model] Validation MAE: {min_mae:.2f} min")

[Minutes Model] Validation R2: 0.7490
[Minutes Model] Validation RMSE: 4.93 min
[Minutes Model] Validation MAE: 3.69 min


In [18]:
importance = pd.Series(xgb_min.feature_importances_, index=MIN_FEATURES)
importance = importance.sort_values(ascending=False)

importance[:50]

STARTING                 0.685933
MIN_ewm10                0.112747
TEAM_MIN_RANK_L10        0.043584
MIN_roll5                0.037941
TEAM_POSS_roll5          0.028900
MIN_share_proxy          0.021065
PF_PER_MIN               0.015463
MIN_season_avg           0.006380
MIN_lag1                 0.005553
MIN_roll10               0.005364
STARTER_ROLL10_PCT       0.004242
STARTING_lag1            0.003967
TEAM_USG_RANK_L10        0.003551
DAYS_REST                0.003363
MEDIAN_MIN_ROLLING_10    0.003079
MIN_TREND                0.002785
MIN_std10                0.002684
IS_B2B                   0.002614
PF_roll5                 0.002455
TEAM_PACE_roll5          0.002300
PLUS_MINUS_roll5         0.002230
BLOWOUT_RISK             0.002196
IS_HOME                  0.001604
dtype: float32

In [ ]:
# from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit


# # Drop rows with NaN in features/target and keep GAME_DATE for time split
# min_df = df[MIN_FEATURES + ["MIN", "GAME_DATE"]].dropna()

# # -- TIME-BASED SPLIT (same style as your new notebook) --
# split_date = min_df["GAME_DATE"].quantile(0.75)  # train on first 75% of season
# train_mask = min_df["GAME_DATE"] <= split_date
# val_mask   = ~train_mask

# X_train_min = min_df.loc[train_mask, MIN_FEATURES]
# y_train_min = min_df.loc[train_mask, "MIN"]
# X_val_min   = min_df.loc[val_mask, MIN_FEATURES]
# y_val_min   = min_df.loc[val_mask, "MIN"]

# param_grid = {
#     "n_estimators": [200, 400, 600, 800],
#     "learning_rate": [0.01, 0.03, 0.05, 0.1],
#     "max_depth": [3, 4, 5, 6, 8],
#     "min_child_weight": [1, 3, 5],
#     "subsample": [0.7, 0.8, 0.9, 1.0],
#     "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
#     "reg_alpha": [0.0, 0.1, 1.0],
#     "reg_lambda": [1.0, 2.0, 5.0],
# }

# xgb_min = XGBRegressor(
#     objective="reg:squarederror",
#     random_state=42,
#     n_jobs=-1
# )

# # Time-aware CV for hyperparameter search on training period only
# tscv = TimeSeriesSplit(n_splits=5)

# search_min = RandomizedSearchCV(
#     estimator=xgb_min,
#     param_distributions=param_grid,
#     n_iter=50,
#     scoring="neg_mean_squared_error",
#     cv=tscv,
#     verbose=1,
#     random_state=42,
#     n_jobs=-1
# )

# search_min.fit(X_train_min, y_train_min)

# print("Best params:", search_min.best_params_)

# min_preds_val = search_min.best_estimator_.predict(X_val_min)

# min_r2 = r2_score(y_val_min, min_preds_val)
# min_rmse = np.sqrt(mean_squared_error(y_val_min, min_preds_val))
# min_mae = mean_absolute_error(y_val_min, min_preds_val)

# print(f"[Minutes Model] Validation R2: {min_r2:.4f}")
# print(f"[Minutes Model] Validation RMSE: {min_rmse:.2f} min")
# print(f"[Minutes Model] Validation MAE: {min_mae:.2f} min")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'subsample': 0.8, 'reg_lambda': 2.0, 'reg_alpha': 1.0, 'n_estimators': 600, 'min_child_weight': 5, 'max_depth': 6, 'learning_rate': 0.01, 'colsample_bytree': 0.7}
[Minutes Model] Validation R2: 0.7490
[Minutes Model] Validation RMSE: 4.93 min
[Minutes Model] Validation MAE: 3.69 min


In [19]:
PPM_FEATURES = [
    "BAYES_PTS_PER_MIN",
    "PTS_roll3", "PTS_lag1", "PTS_ewm10",
    "PTS_PER_MIN_roll3", "PTS_PER_MIN_roll5", "PTS_PER_MIN_roll10",
    "PTS_PER_MIN_10_ewm",
    "USG_PCT_roll3", "USG_PCT_roll5", "USG_PCT_roll10",
    "TS_PCT_roll3", "TS_PCT_roll5", "TS_PCT_roll10",
    "POSS_roll3", "POSS_roll5", "POSS_roll10",
    "TEAM_PACE_roll5", "TEAM_POSS_roll5", "TEAM_POSS_share",
    "MIN_share_proxy", "TEAM_USG_RANK_L10",
    "IS_HOME", "STARTING", "GP", "BLOWOUT_RISK", "PLUS_MINUS_roll3"
]

# Keep only existing columns
PPM_FEATURES = [f for f in PPM_FEATURES if f in df.columns]

ppm_df = (
    df[PPM_FEATURES + ["PTS_PER_MIN", "GAME_DATE"]]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

# -- TIME-BASED SPLIT --
split_date_ppm = ppm_df["GAME_DATE"].quantile(0.75)
train_mask_ppm = ppm_df["GAME_DATE"] <= split_date_ppm
val_mask_ppm   = ~train_mask_ppm

X_train_ppm = ppm_df.loc[train_mask_ppm, PPM_FEATURES]
y_train_ppm = ppm_df.loc[train_mask_ppm, "PTS_PER_MIN"]
X_val_ppm   = ppm_df.loc[val_mask_ppm, PPM_FEATURES]
y_val_ppm   = ppm_df.loc[val_mask_ppm, "PTS_PER_MIN"]

# No scaler needed for XGBoost
xgb_ppm = XGBRegressor(
    n_estimators=400,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_ppm.fit(
    X_train_ppm,
    y_train_ppm,
    eval_set=[(X_val_ppm, y_val_ppm)],
    verbose=False
)

ppm_preds_val = xgb_ppm.predict(X_val_ppm)
ppm_mae = mean_absolute_error(y_val_ppm, ppm_preds_val)
ppm_rmse = np.sqrt(mean_squared_error(y_val_ppm, ppm_preds_val))
ppm_r2 = r2_score(y_val_ppm, ppm_preds_val)

print(f"[Pts/Min Model] Validation R2: {ppm_r2:.4f}")
print(f"[Pts/Min Model] Validation RMSE: {ppm_rmse:.4f} pts/min")
print(f"[Pts/Min Model] Validation MAE: {ppm_mae:.4f} pts/min")

[Pts/Min Model] Validation R2: 0.1965
[Pts/Min Model] Validation RMSE: 0.2606 pts/min
[Pts/Min Model] Validation MAE: 0.1935 pts/min


In [20]:
# Reconstruct val set projection
val_common = ppm_df[val_mask_ppm].copy()
val_common["PRED_PTS_PER_MIN"] = ppm_preds_val
 
val_min_aligned = min_df[val_mask].copy()
val_min_aligned["PRED_MIN"] = min_preds_val
 
# Merge on index (both share original df index ordering)
# Simpler: recompute on shared rows
combined_idx = val_common.index.intersection(val_min_aligned.index)
val_combined = val_common.loc[combined_idx].copy()
val_combined["PRED_MIN"]         = val_min_aligned.loc[combined_idx, "PRED_MIN"]
val_combined["TRUE_PTS"]         = df.loc[combined_idx, "PTS"]
val_combined["PRED_PTS"]         = val_combined["PRED_MIN"] * val_combined["PRED_PTS_PER_MIN"]
 
pts_mae  = mean_absolute_error(val_combined["TRUE_PTS"], val_combined["PRED_PTS"])
pts_bias = (val_combined["PRED_PTS"] - val_combined["TRUE_PTS"]).mean()
print(f"[Points Proj]    Validation MAE:  {pts_mae:.2f} pts")
print(f"[Points Proj]    Bias (pred-true): {pts_bias:.2f} pts")
 

[Points Proj]    Validation MAE:  4.44 pts
[Points Proj]    Bias (pred-true): -0.00 pts


In [23]:
def project_player(player_name: str, opp_def_rating: float = None, spread: float = None):
    """
    Project points for a player's next game.
    opp_def_rating: opponent defensive rating (lower = tougher defense)
    spread: positive = player's team favored
    """
    player_rows = df[df["PLAYER_NAME"].str.lower() == player_name.lower()].copy()
    if player_rows.empty:
        # Try partial match
        mask = df["PLAYER_NAME"].str.lower().str.contains(player_name.lower(), na=False)
        player_rows = df[mask].copy()
    if player_rows.empty:
        return f"Player '{player_name}' not found."

    latest = player_rows.sort_values("GAME_DATE").iloc[-1]

    # -- Minutes projection (XGBoost) --
    min_row = {f: latest.get(f, np.nan) for f in MIN_FEATURES}
    min_row_df = pd.DataFrame([min_row])

    # Fill missing values with training medians (recommended)
    min_fill = X_train_min.median()
    min_row_df = min_row_df.reindex(columns=MIN_FEATURES).fillna(min_fill)

    proj_min = float(xgb_min.predict(min_row_df)[0])

    # Blowout adjustment on minutes
    if spread is not None and abs(spread) > 15:
        proj_min *= 0.90  # -10% for likely blowout

    # -- Pts/Min projection (XGBoost) --
    ppm_row = {f: latest.get(f, np.nan) for f in PPM_FEATURES}
    if opp_def_rating is not None and "OPP_DEF_RATING" in PPM_FEATURES:
        ppm_row["OPP_DEF_RATING"] = opp_def_rating

    ppm_row_df = pd.DataFrame([ppm_row])

    ppm_fill = X_train_ppm.median()
    ppm_row_df = ppm_row_df.reindex(columns=PPM_FEATURES).fillna(ppm_fill)

    proj_ppm = float(xgb_ppm.predict(ppm_row_df)[0])

    proj_pts = proj_min * proj_ppm

    # -- Output --
    print(f"\n{'─'*45}")
    print(f"  Player:          {latest['PLAYER_NAME']}")
    print(f"  Last game:       {latest['GAME_DATE'].date()}  {latest.get('MATCHUP', 'N/A')}  {latest.get('PTS', np.nan)} pts in {latest.get('MIN', np.nan):.1f} min")
    print(f"  Recent avg min:  {latest.get('MIN_roll5', np.nan):.1f}  (last 5 games)")
    print(f"  Starter flag:    {'Yes' if latest.get('STARTING', 0) else 'No'}")
    print(f"  Bayesian ppm:    {latest.get('BAYES_PTS_PER_MIN', np.nan):.3f}")
    print(f"  -- Projections --")
    print(f"  Projected MIN:   {proj_min:.1f}")
    print(f"  Projected ppm:   {proj_ppm:.3f}")
    print(f"  Projected PTS:   {proj_pts:.1f}")
    print(f"{'─'*45}\n")
    return proj_pts


print("\n=== SAMPLE PROJECTIONS (next game) ===")
for name in ["Shai Gilgeous-Alexander"]:
    project_player(name)


=== SAMPLE PROJECTIONS (next game) ===

─────────────────────────────────────────────
  Player:          Shai Gilgeous-Alexander
  Last game:       2026-03-18  OKC @ BKN  20 pts in 25.6 min
  Recent avg min:  36.4  (last 5 games)
  Starter flag:    Yes
  Bayesian ppm:    0.866
  -- Projections --
  Projected MIN:   34.3
  Projected ppm:   0.858
  Projected PTS:   29.4
─────────────────────────────────────────────

